# 🔬 Organoid Segmentation Benchmark
### Training & Evaluating Segmentation Models on Custom RGB + Binary Mask Dataset

**Models benchmarked:**
- Inception-UNet (Andrusca et al., 2026)
- StarDist (Gros et al., 2025)
- DeGPR / YOLOv5-based (Awais et al., 2025)
- DECSEFE-Org / SAM-based (Cicceri et al., 2025)
- microSAM (Archit et al., 2025)
- CellPoseSAM (Pachitariu et al., 2025)
- UNet Baseline

**Metrics tracked:** Loss, IoU (Jaccard Index) & Dice Coefficient  
**Split:** Train 70% / Validation 15% / Test 15%

In [1]:
# Install required packages
!pip install albumentations --quiet
!pip install torch torchvision --quiet
!pip install segmentation-models-pytorch --quiet
!pip install stardist tensorflow --quiet
!pip install cellpose --quiet
!pip install micro-sam --quiet
!pip install opencv-python-headless scikit-learn tqdm pandas matplotlib seaborn --quiet
!pip install torchmetrics --quiet
!pip install "numpy<2.0" --quiet
!pip install --upgrade scipy albumentations -q

print("✅ All packages installed.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

# ─── USER CONFIG ─────────────────────────────────────────────────────────────
CONFIG = {
    # Path to your dataset root.
    # Expected structure:
    #   DATA_ROOT/
    #     images/   ← RGB .png files
    #     masks/    ← Binary B&W .png files (same filename as images)
    "DATA_ROOT": "/kaggle/input/datasets/ngwaru/vascular-organoids-full/dataset",

    "IMAGE_SIZE":   256,    # Resize to (IMAGE_SIZE × IMAGE_SIZE)
    "BATCH_SIZE":    4,
    "NUM_EPOCHS":    10,
    "LEARNING_RATE": 1e-4,

    # Train / Validation / Test split ratios (must sum to 1.0)
    "TRAIN_SPLIT":   0.70,
    "VAL_SPLIT":     0.15,
    "TEST_SPLIT":    0.15,

    "SEED":          42,
    # Use 0 for compatibility on most platforms; increase on multi-core machines
    "NUM_WORKERS":   2,

    # Set to False to skip heavy models (microSAM, CellPoseSAM) on CPU
    "RUN_SAM_MODELS": True,

    "DEVICE":      "cuda",   # 'cuda' or 'cpu'
    "SAVE_DIR":    "./checkpoints",
    "RESULTS_DIR": "./results",     # CSVs land here
}

os.makedirs(CONFIG["SAVE_DIR"],    exist_ok=True)
os.makedirs(CONFIG["RESULTS_DIR"], exist_ok=True)

assert abs(CONFIG["TRAIN_SPLIT"] + CONFIG["VAL_SPLIT"] + CONFIG["TEST_SPLIT"] - 1.0) < 1e-9, \
    "TRAIN_SPLIT + VAL_SPLIT + TEST_SPLIT must equal 1.0"

print("✅ Configuration set.")
print(f"   Data root : {CONFIG['DATA_ROOT']}")
print(f"   Image size: {CONFIG['IMAGE_SIZE']}×{CONFIG['IMAGE_SIZE']}")
print(f"   Epochs    : {CONFIG['NUM_EPOCHS']}")
print(f"   Device    : {CONFIG['DEVICE']}")
print(f"   Split     : {CONFIG['TRAIN_SPLIT']:.0%} train / "
      f"{CONFIG['VAL_SPLIT']:.0%} val / {CONFIG['TEST_SPLIT']:.0%} test")

✅ Configuration set.
   Data root : /kaggle/input/datasets/ngwaru/vascular-organoids-full/dataset
   Image size: 256×256
   Epochs    : 10
   Device    : cuda
   Split     : 70% train / 15% val / 15% test


In [3]:
import torch
import torch.nn as nn
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import shutil


class OrganoidDataset(Dataset):
    """Custom dataset for RGB images + binary masks."""

    def __init__(self, image_dir, mask_dir, img_size=256, augment=False):
        self.image_dir = image_dir
        self.mask_dir  = mask_dir
        self.img_size  = img_size

        self.filenames = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith(".png")
        ])

        if augment:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ElasticTransform(p=0.3),
                A.GaussNoise(p=0.2),
                A.RandomBrightnessContrast(p=0.3),
                A.Normalize(mean=(0.485, 0.456, 0.406),
                            std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ])
        else:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
                A.Normalize(mean=(0.485, 0.456, 0.406),
                            std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]

        img  = cv2.imread(os.path.join(self.image_dir, fname))
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(os.path.join(self.mask_dir, fname),
                          cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)   # binarise

        out = self.transform(image=img, mask=mask)
        return out["image"], out["mask"].unsqueeze(0)


def build_loaders(config):
    """
    Split the full dataset into train / val / test subsets.
    Train uses augmentation; val and test use only normalisation.

    Returns
    -------
    train_loader, val_loader, test_loader
    """
    image_dir = os.path.join(config["DATA_ROOT"], "images")
    mask_dir  = os.path.join(config["DATA_ROOT"], "masks")

    # Build two versions of the dataset — augmented and clean
    aug_ds   = OrganoidDataset(image_dir, mask_dir,
                               img_size=config["IMAGE_SIZE"], augment=True)
    clean_ds = OrganoidDataset(image_dir, mask_dir,
                               img_size=config["IMAGE_SIZE"], augment=False)

    n_total = len(aug_ds)
    n_train = int(n_total * config["TRAIN_SPLIT"])
    n_val   = int(n_total * config["VAL_SPLIT"])
    n_test  = n_total - n_train - n_val   # absorbs rounding remainder

    torch.manual_seed(config["SEED"])
    # Random split on augmented dataset to get the index partition
    train_ds_aug, val_ds_tmp, test_ds_tmp = random_split(
        aug_ds, [n_train, n_val, n_test]
    )

    # Val and test subsets re-index into the *clean* (no-aug) dataset
    val_ds  = Subset(clean_ds, val_ds_tmp.indices)
    test_ds = Subset(clean_ds, test_ds_tmp.indices)


    train_filenames = [clean_ds.filenames[i] for i in train_ds_aug.indices]
    val_filenames   = [clean_ds.filenames[i]for i in val_ds_tmp.indices]
    test_filenames  = [clean_ds.filenames[i] for i in test_ds_tmp.indices]
    
    # 5. Convert lists to DataFrames and export to 3 separate CSV files
    pd.DataFrame({"image_filename": train_filenames}).to_csv("results/train_data.csv", index=False)
    pd.DataFrame({"image_filename": val_filenames}).to_csv("results/val_data.csv", index=False)
    pd.DataFrame({"image_filename": test_filenames}).to_csv("results/test_data.csv", index=False)
    
    print(f"Created train_data.csv, val_data.csv, and test_data.csv successfully. {len(train_filenames), len(val_filenames), len(test_filenames)} ")




    

    nw = config["NUM_WORKERS"]
    train_loader = DataLoader(train_ds_aug, batch_size=config["BATCH_SIZE"],
                              shuffle=True,  num_workers=nw, pin_memory=True)
    val_loader   = DataLoader(val_ds,       batch_size=config["BATCH_SIZE"],
                              shuffle=False, num_workers=nw, pin_memory=True)
    test_loader  = DataLoader(test_ds,      batch_size=config["BATCH_SIZE"],
                              shuffle=False, num_workers=nw, pin_memory=True)

    print(f"✅ Dataset split — Train: {n_train} | Val: {n_val} | Test: {n_test} "
          f"(Total: {n_total})")
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders(CONFIG)

Created train_data.csv, val_data.csv, and test_data.csv successfully. (401, 85, 87) 
✅ Dataset split — Train: 401 | Val: 85 | Test: 87 (Total: 573)


In [4]:
main_path = CONFIG["DATA_ROOT"]
train_df = pd.read_csv("results/train_data.csv")
val_df   = pd.read_csv("results/val_data.csv")
test_df  = pd.read_csv("results/test_data.csv")

# target size (adjust as needed)
TARGET_SIZE = (CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])

os.makedirs("cellpose_data", exist_ok=True)

for df, folder_name in zip([train_df, val_df, test_df], ["train", "val", "test"]):
    dest_path = os.path.join("cellpose_data", folder_name)
    os.makedirs(dest_path, exist_ok=True)

    og_path_img = os.path.join(main_path, "images")
    og_path_msk = os.path.join(main_path, "masks")   # <- corrected, masks folder not images

    for file_name in df['image_filename']:
        # --- resize image ---
        img = cv2.imread(os.path.join(og_path_img, file_name), cv2.IMREAD_COLOR)
        img_resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        cv2.imwrite(os.path.join(dest_path, f"{file_name.replace('.png','')}_img.png"), img_resized)

        # --- resize mask ---
        mask = cv2.imread(os.path.join(og_path_msk, file_name), cv2.IMREAD_GRAYSCALE)
        mask_resized = cv2.resize(mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(os.path.join(dest_path, f"{file_name.replace('.png','')}_mask.png"), mask_resized)

In [5]:
# train_filenames = [train_loader.filenames[i] for i in intrain_loader]
# val_filenames   = [dataset.filenames[i] for i in val_loader.indices]
# test_filenames  = [dataset.filenames[i] for i in test_loader.indices]

# # 5. Convert lists to DataFrames and export to 3 separate CSV files
# pd.DataFrame({"image_filename": train_filenames}).to_csv("train_data.csv", index=False)
# pd.DataFrame({"image_filename": val_filenames}).to_csv("val_data.csv", index=False)
# pd.DataFrame({"image_filename": test_filenames}).to_csv("test_data.csv", index=False)

# print("Created train_data.csv, val_data.csv, and test_data.csv successfully.")

In [6]:
import torch.nn.functional as F
import segmentation_models_pytorch as smp


# ─────────────────────────────────────────────────────────────────────────────
# 4A. Inception-UNet  (Andrusca et al., 2026)
# ─────────────────────────────────────────────────────────────────────────────
class InceptionBlock(nn.Module):
    """Parallel 1×1 / 3×3 / 5×5 convolutions concatenated."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 4
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch5 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 5, padding=2), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.pool    = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.proj    = nn.Sequential(
            nn.Conv2d(mid * 4, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.proj(
            torch.cat([self.branch1(x), self.branch3(x),
                       self.branch5(x), self.pool(x)], dim=1))


class InceptionUNet(nn.Module):
    """UNet with Inception encoder blocks."""

    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            InceptionBlock(in_ch, out_ch),
            InceptionBlock(out_ch, out_ch))

    def __init__(self, in_ch=3, out_ch=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.pool     = nn.MaxPool2d(2)
        self.encoders = nn.ModuleList()
        prev = in_ch
        for f in features:
            self.encoders.append(self._block(prev, f))
            prev = f

        self.bottleneck = self._block(features[-1], features[-1] * 2)

        self.ups      = nn.ModuleList()
        self.decoders = nn.ModuleList()
        in_f = features[-1] * 2
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(in_f, f, 2, 2))
            self.decoders.append(self._block(f * 2, f))
            in_f = f

        self.final = nn.Conv2d(features[0], out_ch, 1)

    def forward(self, x):
        skips, out = [], x
        for enc in self.encoders:
            out = enc(out); skips.append(out); out = self.pool(out)
        out = self.bottleneck(out)
        for up, dec, skip in zip(self.ups, self.decoders, reversed(skips)):
            out = up(out)
            if out.shape != skip.shape:
                out = F.interpolate(out, size=skip.shape[2:])
            out = dec(torch.cat([skip, out], dim=1))
        return self.final(out)


# ─────────────────────────────────────────────────────────────────────────────
# 4B. Baseline UNet  (Present Thesis)
# ─────────────────────────────────────────────────────────────────────────────
def build_unet():
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet",
                    in_channels=3, classes=1)


# ─────────────────────────────────────────────────────────────────────────────
# 4C. DeGPR / YOLOv5-style  (Awais et al., 2025)
# ─────────────────────────────────────────────────────────────────────────────
def build_degpr():
    return smp.FPN(encoder_name="resnet18", encoder_weights="imagenet",
                   in_channels=3, classes=1)


# ─────────────────────────────────────────────────────────────────────────────
# 4D. DECSEFE-Org  (Cicceri et al., 2025)
# ─────────────────────────────────────────────────────────────────────────────
def build_decsefe():
    return smp.UnetPlusPlus(encoder_name="densenet121", encoder_weights="imagenet",
                            in_channels=3, classes=1)


# ─────────────────────────────────────────────────────────────────────────────
# 4E. microSAM  (Archit et al., 2025)
# Falls back to UNet surrogate when micro-sam is unavailable.
# ─────────────────────────────────────────────────────────────────────────────
class MicroSAMWrapper(nn.Module):

    def __init__(self, img_size=256):
        super().__init__()
        try:
            from micro_sam.training import get_sam_model
            self.model   = get_sam_model(model_type="vit_b")
            self._native = True
            print("  micro-SAM loaded (vit_b).")
        except Exception as e:
            print(f"  ⚠️  micro-SAM unavailable ({e}). Using UNet surrogate.")
            self.model   = build_unet()
            self._native = False

    def forward(self, x):
        if self._native:
            preds, _, _ = self.model(x, multimask_output=False)
            return preds
        return self.model(x)


# ─────────────────────────────────────────────────────────────────────────────
# 4F. CellPoseSAM  (Pachitariu et al., 2025)
# Falls back to MAnet surrogate when cellpose is unavailable.
# NOTE: native CellPose runs as inference-only (no gradient training).
# ─────────────────────────────────────────────────────────────────────────────






def CellposeTrainer(n_epochs, lr):
    
    from cellpose import models, train
    from cellpose import io as cellpose_io
    cellpose_io.logger_setup()
    train_dir = "cellpose_data/train"
    test_dir = "cellpose_data/val"
    output = cellpose_io.load_train_test_data(train_dir, test_dir, image_filter="_img",
                                    mask_filter="_mask", look_one_level_down=False)
    images, labels, image_names, test_images, test_labels, image_names_test = output
    
    model = models.CellposeModel(gpu=True, model_type='cpsam')
    
    model_path, train_losses, test_losses = train.train_seg(model.net,
                                train_data=images, train_labels=labels,
                                test_data=test_images, test_labels=test_labels,
                            weight_decay=0.1, learning_rate=lr,
                            min_train_masks=0,
                            n_epochs=n_epochs, model_name="cellpose_model")
    return model_path, train_losses, test_losses






class CellPoseSAMWrapper(nn.Module):

    def __init__(self, img_size=256):
        super().__init__()
        try:
            from cellpose import models as cp_models
           
            train_loss, val_loss = CellposeWrapper()
            history = []
            best_loss = np.inf
            for idx, (tr_ls, va_ls) in enumerate(zip(train_loss, val_loss)):
                if best_loss < tr_ls:
                    best_loss = tr_ls
                history.append({
                "epoch":      idx,
                "train_loss": tr_ls,
                "val_loss":   va_ls,
                "val_iou":   0,
                "val_dice":  0,
                })
            self.best_loss = best_loss
            self.history = history
            self.cp = cp_models.CellposeModel(gpu=torch.cuda.is_available(), pretrained_model="cellpose_model")
            
            # image = skio.imread(image_path)
            # image = transform.resize(image, (512, 512), anti_aliasing=True)
            # Run prediction to get semantic masks refined by SAM's prompt geometry
            # masks, flows, styles = model.eval(
            #     image,
            #     diameter=None,
            #     channels=[0, 0], # Change based on your channels (e.g., [cytoplasm, nucleus])
                # logit_threshold=0.0,
                # amsgrad=True
            # )
            
            # self.cp      = cp_models.CellposeModel(
            #     gpu=torch.cuda.is_available(), model_type="cyto3")
            self._native = True
           
            print("  CellposeSAM loaded.")
        except Exception as e:
            print(f"  ⚠️  CellposeSAM unavailable ({e}). Using MAnet surrogate.")
            self.model   = smp.MAnet(encoder_name="resnet34", encoder_weights="imagenet",
                                     in_channels=3, classes=1)
            self._native = False

    def forward(self, x):
        if self._native:
            
            imgs_np = x.permute(0, 2, 3, 1).cpu().numpy()
            if train_loop:
                masks, _, _ = self.cp.net(imgs_np, diameter=None, channels=[0, 0])
            else:
                masks, _, _ = self.cp.eval(imgs_np, diameter=None, channels=[0, 0])
            masks_t = torch.from_numpy(np.stack([(m > 0).astype(np.float32) for m in masks])).unsqueeze(1).to(x.device)
            return masks_t
        return self.model(x)



# class CellPoseSAMWrapper(nn.Module):
#     def __init__(
#         self,
#         gpu=True,
#         model_type="cyto3",
#         surrogate_encoder="resnet34"
#     ):
#         super().__init__()

#         try:
#             from cellpose import models

#             self.cp_model = models.CellposeModel(
#                 gpu=gpu and torch.cuda.is_available(),
#                 model_type=model_type
#             )

#             # Cellpose network used by train.train_seg()
#             self.net = self.cp_model.net
#             self._native = True

#             print("Cellpose loaded successfully.")

#         except Exception as e:
#             print(f"Cellpose unavailable ({e}). Falling back to MAnet.")

#             import segmentation_models_pytorch as smp

#             self.net = smp.MAnet(
#                 encoder_name=surrogate_encoder,
#                 encoder_weights="imagenet",
#                 in_channels=3,
#                 classes=1,
#             )

#             self._native = False

#     def forward(self, x):
#         """
#         Inference.
#         x: (B,C,H,W) torch tensor
#         """
#         if self._native:
#             imgs_np = x.detach().permute(0, 2, 3, 1).cpu().numpy()

#             masks, _, _ = self.cp_model.eval(
#                 imgs_np,
#                 diameter=None,
#                 channels=[0, 0]
#             )

#             masks = np.stack(
#                 [(m > 0).astype(np.float32) for m in masks]
#             )

#             return torch.from_numpy(masks).unsqueeze(1).to(x.device)

#         return self.net(x)

#     def fit(
#         self,
#         train_images,
#         train_labels,
#         test_images="_img",
#         test_labels="_mask",
#         n_epochs=30,
#         learning_rate=1e-5,
#         weight_decay=0.1,
#         model_name="vascular_model",
#         min_train_masks=0,
#     ):
#         """
#         Train Cellpose using the official Cellpose API.
#         """

#         if not self._native:
#             raise RuntimeError(
#                 "Cellpose training unavailable because "
#                 "the native Cellpose model could not be loaded."
#             )

#         from cellpose import train

#         model_path, train_losses, test_losses = train.train_seg(
#             self.net,
#             train_data=train_images,
#             train_labels=train_labels,
#             test_data=test_images,
#             test_labels=test_labels,
#             learning_rate=learning_rate,
#             weight_decay=weight_decay,
#             n_epochs=n_epochs,
#             model_name=model_name,
#             min_train_masks=min_train_masks,
#         )

#         return {
#             "model_path": model_path,
#             "train_losses": train_losses,
#             "test_losses": test_losses,
#         }



# ─────────────────────────────────────────────────────────────────────────────
# 4G. StarDist  (Gros et al., 2025)
# Falls back to LinkNet surrogate when stardist is unavailable.
# NOTE: native StarDist runs as inference-only (no gradient training).
# ─────────────────────────────────────────────────────────────────────────────
class StarDistWrapper(nn.Module):

    def __init__(self, img_size=256):
        super().__init__()
        try:
            from stardist.models import StarDist2D
            self.sd      = StarDist2D.from_pretrained("2D_versatile_fluo")
            self._native = True
            print("  StarDist2D loaded.")
        except Exception as e:
            print(f"  ⚠️  StarDist unavailable ({e}). Using LinkNet surrogate.")
            self.model   = smp.Linknet(encoder_name="resnet18", encoder_weights="imagenet",
                                       in_channels=3, classes=1)
            self._native = False

    def forward(self, x):
        if self._native:
            preds = []
            for img in x.permute(0, 2, 3, 1).cpu().numpy():
                # img_u8 = ((img - img.min()) / (img.ptp() + 1e-6) * 255).astype(np.uint8)
                img_u8  ((img - img.min()) / (np.ptp(img) + 1e-6) * 255).astype(np.uint8)

                gray   = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY)
                labels, _ = self.sd.predict_instances(gray.astype(np.float32) / 255.0)
                preds.append((labels > 0).astype(np.float32))
            return torch.from_numpy(np.stack(preds)).unsqueeze(1).to(x.device)
        return self.model(x)


def get_trainable_parameters(model):
    """
    Return an iterable of parameters that support gradient computation.
    For native inference wrappers (CellPose, StarDist) that expose no
    PyTorch parameters, returns None — the caller must skip optimisation.
    """
    try:
        params = list(model.parameters())
    except Exception:
        return None
    if not params:
        return None
    return params





print("✅ All model classes defined.")

✅ All model classes defined.


In [7]:
class CombinedLoss(nn.Module):
    """BCE + Dice loss.  alpha controls weighting."""

    def __init__(self, alpha=0.5, smooth=1e-6):
        super().__init__()
        self.alpha  = alpha
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()

    def dice_loss(self, logits, targets):
        preds = torch.sigmoid(logits).view(-1)
        tgts  = targets.view(-1)
        num   = (preds * tgts).sum()
        den   = preds.sum() + tgts.sum()
        return 1.0 - (2.0 * num + self.smooth) / (den + self.smooth)

    def forward(self, logits, targets):
        return (self.alpha * self.bce(logits, targets)
                + (1 - self.alpha) * self.dice_loss(logits, targets))


def compute_iou(preds, targets, threshold=0.5, smooth=1e-6):
    """Binary IoU (Jaccard) over a batch."""
    if isinstance(preds, list):
        preds = np.stack(preds)
    if isinstance(preds, np.ndarray):
        preds = torch.from_numpy(preds)
    if isinstance(targets, np.ndarray):
        targets = torch.from_numpy(targets)
    p = (preds > threshold).float().view(-1)
    t = targets.view(-1)
    inter = (p * t).sum()
    union = p.sum() + t.sum() - inter
    return ((inter + smooth) / (union + smooth)).item()


def compute_dice(preds, targets, threshold=0.5, smooth=1e-6):
    """Binary Dice coefficient over a batch."""
    if isinstance(preds, list):
        preds = np.stack(preds)
    if isinstance(preds, np.ndarray):
        preds = torch.from_numpy(preds)
    if isinstance(targets, np.ndarray):
        targets = torch.from_numpy(targets)
    p = (preds > threshold).float().view(-1)
    t = targets.view(-1)
    inter = (p * t).sum()
    return ((2.0 * inter + smooth) / (p.sum() + t.sum() + smooth)).item()


print("✅ Loss function and metrics ready.")

✅ Loss function and metrics ready.


In [8]:
from tqdm.notebook import tqdm
import pandas as pd
import time

def save_mask_func(out, name):
    new_name = name.replace(" ", "_")
    os.makedirs(f"{new_name}_mask_predictions", exist_ok=True)
    df_test = pd.read_csv("results/test_data.csv")
    for idx, (mask, img_filename) in enumerate(zip(out, df_test['image_filename'])):
        # Ensure mask is uint8 (0–255)
        mask_img = (mask.astype("uint8")) * 255
        
        # Save as PNG
        save_path = os.path.join("cellpose_mask_predictions", f"{img_filename}")
        cv2.imwrite(save_path, mask_img)
    
    print(f"Saved masks")


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        if out.shape != masks.shape:
            out = F.interpolate(out, size=masks.shape[2:])
        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device, criterion=None, name="UNet", save_mask=False):
    """
    Evaluate model on a DataLoader.

    Returns
    -------
    avg_loss : float  (nan if criterion is None)
    avg_iou  : float
    avg_dice : float
    """
    model.eval()
    ious, dices, losses = [], [], []
    _crit = criterion or CombinedLoss(alpha=0.5)
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        out = model(imgs)
        if out.shape != masks.shape:
            out = F.interpolate(out, size=masks.shape[2:])
        if save_mask:
            save_mask_func(out, name)
        losses.append(_crit(out, masks).item())
        preds = torch.sigmoid(out)
        ious.append(compute_iou(preds.cpu(),  masks.cpu()))
        dices.append(compute_dice(preds.cpu(), masks.cpu()))
    return float(np.mean(losses)), float(np.mean(ious)), float(np.mean(dices))


def train_model(name, model, config, train_loader, val_loader):
    """
    Full training loop.

    Handles models that have no trainable PyTorch parameters
    (native CellPose / StarDist wrappers) by running them in
    inference-only mode and recording metrics without optimisation.

    Returns
    -------
    best_loss : float
    best_iou  : float
    best_dice : float
    history   : list[dict]  — one dict per epoch
    """
    print(f"\n{'═'*60}")
    print(f"  Training: {name}")
    print(f"{'═'*60}")

    device    = torch.device(config["DEVICE"] if torch.cuda.is_available() else "cpu")
    model     = model.to(device)
    criterion = CombinedLoss(alpha=0.5)
    # if name == "CellPoseSAM":
    #     train_loss, val_loss = CellposeWrapper()
    #     history = []
    #     best_loss = np.inf
    #     for idx, (tr_ls, va_ls) in enumerate(zip(train_loss, val_loss)):
    #         if best_loss < tr_ls:
    #             best_loss = tr_ls
    #         history.append({
    #         "epoch":      idx,
    #         "train_loss": tr_ls,
    #         "val_loss":   va_ls,
    #         "val_iou":    ,
    #         "val_dice":   ,
    #         })
    #     return best_loss, best_iou, best_dice, history

    # ── Detect trainable parameters ──────────────────────────────────────────
    params = get_trainable_parameters(model)
    if params is None:
        print("  ℹ️  No trainable PyTorch parameters found — inference-only mode.")
        inference_only = True
        optimizer = scheduler = None
    else:
        inference_only = False
        optimizer = torch.optim.AdamW(params, lr=config["LEARNING_RATE"],
                                      weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=config["NUM_EPOCHS"], eta_min=1e-6)

    best_loss, best_iou, best_dice = float("inf"), 0.0, 0.0
    history = []
    epochs = config["NUM_EPOCHS"]
    if name == "CellPoseSAM":
        epochs = 3
    for epoch in tqdm(range(1, epochs + 1), desc=name):
        
        if inference_only:
            # No gradient step; just record validation metrics
            train_loss = float("nan")
        else:
            train_loss = train_one_epoch(model, train_loader, optimizer,
                                         criterion, device)

        val_loss, val_iou, val_dice = evaluate(model, val_loader, device, criterion)

        if not inference_only:
            scheduler.step()

        history.append({
            "epoch":      epoch,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_iou":    val_iou,
            "val_dice":   val_dice,
        })

        # Track best checkpoint by validation IoU
        if val_iou > best_iou:
            best_loss, best_iou, best_dice = val_loss, val_iou, val_dice
            os.makedirs(config["SAVE_DIR"], exist_ok=True)
            ckpt_path = os.path.join(
                config["SAVE_DIR"], f"{name.replace(' ', '_')}_best.pt")
            torch.save(model.state_dict(), ckpt_path)
    
        if name == "CellPoseSAM":
            for idx_i in zip(history,model.history):
                history[idx_i]['train_loss'] = model.history[idx_i]['train_loss']
                history[idx_i]['val_loss'] = model.history[idx_i]['val_loss']
            
            print(f"  Epoch {epoch:3d}/{config['NUM_EPOCHS']} | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | "
                  f"IoU: {val_iou:.4f} | Dice: {val_dice:.4f}")
            
            best_loss = model.best_loss
        else:
            if epoch % 5 == 0 or epoch == 1:
                print(f"  Epoch {epoch:3d}/{config['NUM_EPOCHS']} | "
                      f"Train Loss: {train_loss:.4f} | "
                      f"Val Loss: {val_loss:.4f} | "
                      f"IoU: {val_iou:.4f} | Dice: {val_dice:.4f}")


    print(f"  ✅ Best — Loss: {best_loss:.4f}  IoU: {best_iou:.4f}  Dice: {best_dice:.4f}")
    return best_loss, best_iou, best_dice, history


print("✅ Training engine ready.")

✅ Training engine ready.


In [9]:
# Load Hugging Face token (Kaggle environment only)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HF_TOKEN loaded from Kaggle secrets.")
except Exception as e:
    print(f"ℹ️  Kaggle secrets not available ({e}). "
          "Set HF_TOKEN manually if required by micro-sam.")

✅ HF_TOKEN loaded from Kaggle secrets.


In [10]:
!nvidia-smi

results   = []
histories = {}

# ── Model registry ────────────────────────────────────────────────────────────
# Each entry: (display_name, author_ref, factory_fn)
model_registry = [
    ("Inception-UNet",
     "Andrusca et al., 2026",
     lambda: InceptionUNet(in_ch=3, out_ch=1)),

    ("UNet (Baseline)",
     "Present Thesis",
     build_unet),

    # ("DeGPR / FPN-ResNet18",
    #  "Awais et al., 2025",
    #  build_degpr),

    # ("DECSEFE-Org / UNet++ DenseNet",
    #  "Cicceri et al., 2025",
    #  build_decsefe),
]

if CONFIG["RUN_SAM_MODELS"]:
    model_registry += [
        # ("StarDist",
        #  "Gros et al., 2025",
        #  lambda: StarDistWrapper(CONFIG["IMAGE_SIZE"])),

        ("microSAM",
         "Archit et al., 2025",
         lambda: MicroSAMWrapper(CONFIG["IMAGE_SIZE"])),

        # ("CellPoseSAM",
        #  "Pachitariu et al., 2025",
        #  lambda: CellPoseSAMWrapper()),
    ]


# ── Train each model ──────────────────────────────────────────────────────────
for name, author, factory in model_registry:
    t0 = time.time()
    try:
        model = factory()
        best_loss, best_iou, best_dice, hist = train_model(
            name, model, CONFIG, train_loader, val_loader)

        # ── Per-model CSV ─────────────────────────────────────────────────────
        hist_df = pd.DataFrame(hist)
        hist_df.insert(0, "model", name)
        safe_name = name.replace(" ", "_").replace("/", "-")
        per_model_csv = os.path.join(CONFIG["RESULTS_DIR"],
                                     f"{safe_name}_history.csv")
        hist_df.to_csv(per_model_csv, index=False)
        print(f"  📄 Per-model CSV saved → {per_model_csv}")

        results.append({
            "Model":               name,
            "Author / Reference":  author,
            "Best Val Loss":       round(best_loss, 4),
            "Best Val IoU":        round(best_iou,  4),
            "Best Val Dice":       round(best_dice, 4),
        })
        histories[name] = hist

    except Exception as exc:
        import traceback
        print(f"  ❌ Failed to train {name}: {exc}")
        traceback.print_exc()
        results.append({
            "Model":               name,
            "Author / Reference":  author,
            "Best Val Loss":       None,
            "Best Val IoU":        None,
            "Best Val Dice":       None,
        })
    finally:
        # Free VRAM regardless of success / failure
        try:
            del model
        except NameError:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  ⏱  {name} — {time.time() - t0:.1f}s")

print("\n✅ All models trained.")

Tue Jun  2 14:18:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Inception-UNet:   0%|          | 0/10 [00:00<?, ?it/s]

  Epoch   1/10 | Train Loss: 0.6153 | Val Loss: 0.4603 | IoU: 0.9179 | Dice: 0.9571
  Epoch   5/10 | Train Loss: 0.3109 | Val Loss: 0.2653 | IoU: 0.9474 | Dice: 0.9729
  Epoch  10/10 | Train Loss: 0.2474 | Val Loss: 0.2245 | IoU: 0.9534 | Dice: 0.9761
  ✅ Best — Loss: 0.2300  IoU: 0.9536  Dice: 0.9761
  📄 Per-model CSV saved → ./results/Inception-UNet_history.csv
  ⏱  Inception-UNet — 570.2s

════════════════════════════════════════════════════════════
  Training: UNet (Baseline)
════════════════════════════════════════════════════════════


UNet (Baseline):   0%|          | 0/10 [00:00<?, ?it/s]

  Epoch   1/10 | Train Loss: 0.4589 | Val Loss: 0.2887 | IoU: 0.9187 | Dice: 0.9576
  Epoch   5/10 | Train Loss: 0.1030 | Val Loss: 0.0865 | IoU: 0.9566 | Dice: 0.9777
  Epoch  10/10 | Train Loss: 0.0653 | Val Loss: 0.0575 | IoU: 0.9674 | Dice: 0.9834
  ✅ Best — Loss: 0.0580  IoU: 0.9674  Dice: 0.9834
  📄 Per-model CSV saved → ./results/UNet_(Baseline)_history.csv
  ⏱  UNet (Baseline) — 404.3s
  ⚠️  micro-SAM unavailable (No module named 'micro_sam'). Using UNet surrogate.

════════════════════════════════════════════════════════════
  Training: microSAM
════════════════════════════════════════════════════════════


microSAM:   0%|          | 0/10 [00:00<?, ?it/s]

  Epoch   1/10 | Train Loss: 0.4643 | Val Loss: 0.2908 | IoU: 0.9323 | Dice: 0.9649
  Epoch   5/10 | Train Loss: 0.1324 | Val Loss: 0.1018 | IoU: 0.9638 | Dice: 0.9815
  Epoch  10/10 | Train Loss: 0.0803 | Val Loss: 0.0689 | IoU: 0.9665 | Dice: 0.9829
  ✅ Best — Loss: 0.0765  IoU: 0.9665  Dice: 0.9829
  📄 Per-model CSV saved → ./results/microSAM_history.csv
  ⏱  microSAM — 388.6s

✅ All models trained.


In [11]:
# def cellpose_inference(image_path, cp_model_path):
#     from cellpose import models
#     from cellpose import io as skio
    
    
#     model = models.CellposeModel(model_type='cpsam', gpu=True)
#     # images = io.load_images([image])
#     if isinstance(image_path, list):
#         image = [np.transpose(skio.imread(p), (2, 0, 1)) for p in image_path]
#     else:
#         image = skio.imread(image_path)
#         image = np.transpose(image, (2, 0, 1))

#     # image = transform.resize(image, (512, 512), anti_aliasing=True)
#     # Run prediction to get semantic masks refined by SAM's prompt geometry
#     masks, flows, styles = model.eval(
#         image,
#         diameter=None,
#         channels=[0, 0], # Change based on your channels (e.g., [cytoplasm, nucleus])
#         channel_axis=0
#         # logit_threshold=0.0,
#         # amsgrad=True
#     )
#     return masks, flows, styles


In [12]:
# def sigmoid(x):
#     return 1 / (1 + np.exp(-x))


def cellpose_inference(image_path, cp_model_path):
    from cellpose import models
    from cellpose import io as skio
    
    
    model = models.CellposeModel(gpu=True, pretrained_model=cp_model_path)
    # images = io.load_images([image])
    if isinstance(image_path, list):
        image = [np.transpose(skio.imread(p), (2, 0, 1)) for p in image_path]
    else:
        image = skio.imread(image_path)
        image = np.transpose(image, (2, 0, 1))

    # image = transform.resize(image, (512, 512), anti_aliasing=True)
    # Run prediction to get semantic masks refined by SAM's prompt geometry
    masks, flows, styles = model.eval(
        image,
        diameter=None,
        channels=[0, 0], # Change based on your channels (e.g., [cytoplasm, nucleus])
        channel_axis=0
        # logit_threshold=0.0,
        # amsgrad=True
    )
    return masks, flows, styles

cellposesam_results = []

def train_cellpose(name, n_epochs, lr):
    t0 = time.time()
    cp_model_path, cp_train_loss, cp_test_loss = CellposeTrainer(n_epochs, lr)
    cellposesam_results.append((cp_model_path, cp_train_loss, cp_test_loss))
    history = []
    for i, j in enumerate(zip(cp_train_loss, cp_test_loss)):
         history.append({
            "epoch":      i,
            "train_loss": j[0],
            "val_loss":   j[1],
            "val_iou":    0,
            "val_dice":   0,
        })

        # ── Per-model CSV ─────────────────────────────────────────────────────
    hist_df = pd.DataFrame(hist)
    hist_df.insert(0, "model", name)
    safe_name = name.replace(" ", "_").replace("/", "-")
    per_model_csv = os.path.join(CONFIG["RESULTS_DIR"],
                                 f"{safe_name}_history.csv")
    hist_df.to_csv(per_model_csv, index=False)
    print(f"  📄 Per-model CSV saved → {per_model_csv}")
    df = pd.read_csv("results/val_data.csv")

    img_path_list = []
    msk_list = []
    temp_img_list_path =[]
    
    for test_image in df['image_filename']:
        image_path = os.path.join(f'{CONFIG["DATA_ROOT"]}/images', test_image)
        mask_path = os.path.join(f'{CONFIG["DATA_ROOT"]}/masks', test_image)
        img_path_list.append(image_path)
    
        msk_list.append(cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), (CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE'])))

    if os.path.exists("temp_dir"):
        shutil.rmtree("temp_dir")
    os.makedirs("temp_dir", exist_ok=True)
    for img in img_path_list:
        temp_img = cv2.resize(cv2.imread(img), (CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE']))
        temp_img_list_path.append(os.path.join("temp_dir",img.split("/")[-1]))
        cv2.imwrite(os.path.join("temp_dir",img.split("/")[-1]), temp_img)


    
        
    multiple_masks, _, _ = cellpose_inference(temp_img_list_path, cp_model_path)
    
    preds = multiple_masks
    targets = np.array(msk_list)
    preds = np.array(multiple_masks)
    targets = np.array(msk_list)
        
    preds_bin   = (np.array(multiple_masks) > 0).astype(np.float32)
    targets_bin = (np.array(msk_list) > 127).astype(np.float32)

        
    cp_best_iou = compute_iou(preds_bin, targets_bin, threshold=0.5, smooth=1e-6)
    cp_best_dice = compute_dice(preds_bin, targets_bin, threshold=0.5, smooth=1e-6)
    cp_best_loss = np.mean(cp_test_loss)

    

    results.append({
            "Model":               name,
            "Author / Reference":  "Pachitariu et al., 2025",
            "Best Val Loss":       round(cp_best_loss, 4),
            "Best Val IoU":        round(cp_best_iou,  4),
            "Best Val Dice":       round(cp_best_dice, 4),
        })

    histories[name] = history
    print(f"  ⏱  {name} — {time.time() - t0:.1f}s")


train_cellpose("CellPoseSAM", CONFIG["NUM_EPOCHS"],CONFIG["LEARNING_RATE"])

2026-06-02 14:41:47,406 [INFO] WRITING LOG OUTPUT TO /root/.cellpose/run.log
2026-06-02 14:41:47,407 [INFO] 
cellpose version: 	4.1.1 
platform:       	linux 
python version: 	3.12.13 
torch version:  	2.10.0+cu128
2026-06-02 14:41:47,440 [INFO] not all flows are present, running flow generation for all images
2026-06-02 14:41:48,329 [INFO] 401 / 401 images in cellpose_data/train folder have labels
2026-06-02 14:41:48,338 [INFO] not all flows are present, running flow generation for all images
2026-06-02 14:41:48,528 [INFO] 85 / 85 images in cellpose_data/val folder have labels
2026-06-02 14:41:48,529 [WARNING] model_type argument is not used in v4.0.1+. Ignoring this argument...
2026-06-02 14:41:48,531 [INFO] ** TORCH CUDA version installed and working. **
2026-06-02 14:41:48,532 [INFO] >>>> using GPU (CUDA)
2026-06-02 14:41:51,038 [INFO] >>>> loading model /root/.cellpose/models/cpsam
2026-06-02 14:41:51,774 [INFO] >>> converting bfloat16 network to float32 for training
2026-06-02 14

100%|██████████| 401/401 [00:27<00:00, 14.85it/s]

2026-06-02 14:42:18,831 [INFO] computing flows for labels



100%|██████████| 85/85 [00:06<00:00, 13.48it/s]

2026-06-02 14:42:25,151 [INFO] >>> computing diameters



100%|██████████| 85/85 [00:00<00:00, 3007.86it/s]

2026-06-02 14:42:25,316 [INFO] >>> normalizing {'lowhigh': None, 'percentile': None, 'normalize': True, 'norm3D': True, 'sharpen_radius': 0, 'smooth_radius': 0, 'tile_norm_blocksize': 0, 'tile_norm_smooth3D': 1, 'invert': False}


2026-06-02 14:42:27,349 [INFO] >>> n_epochs=10, n_train=401, n_test=85
2026-06-02 14:42:27,350 [INFO] >>> AdamW, learning_rate=0.00010, weight_decay=0.10000
2026-06-02 14:42:27,352 [INFO] >>> saving model to /kaggle/working/models/cellpose_model
2026-06-02 14:49:02,043 [INFO] 0, train_loss=0.0559, test_loss=0.0674, LR=0.000000, time 394.69s
2026-06-02 15:20:51,107 [INFO] 5, train_loss=0.0256, test_loss=0.0289, LR=0.000056, time 2303.75s
2026-06-02 15:45:56,022 [INFO] saving network parameters to /kaggle/working/models/cellpose_model
2026-06-02 15:46:00,158 [INFO] >>> converting network back to torch.bfloat16 after training
  📄 Per-model CSV saved → ./results/CellPoseSAM_history.csv
2026-06-02 15:46:12,695 [INFO] ** TORCH CUDA version installed and working. **
2026-06-02 15:46:12,695 [INFO] >>>> using GPU (CUDA)
2026-06-02 15:46:14,696 [INFO] >>>> loading model /kaggle/working/models/cellpose_model
2026-06-02 15:46:15,748 [WARNING] channels deprecated in v4.0.1+. If data contain more th

In [ ]:
# ── Evaluate each successfully trained model on the held-out test set ────────
print("\n" + "═"*60)
print("  TEST SET EVALUATION")
print("═"*60)
import shutil

test_results = []
name_to_factory = {r[0]: r[2] for r in model_registry}
criterion = CombinedLoss(alpha=0.5)
device    = torch.device(CONFIG["DEVICE"] if torch.cuda.is_available() else "cpu")

for entry in results:
    name = entry["Model"]
    if name == "CellPoseSAM":
        try:
            df = pd.read_csv("results/test_data.csv")
    
            img_path_list = []
            msk_list = []
            temp_img_list_path =[]
            for test_image in df['image_filename']:
                image_path = os.path.join(f'{CONFIG["DATA_ROOT"]}/images', test_image)
                mask_path = os.path.join(f'{CONFIG["DATA_ROOT"]}/masks', test_image)
                img_path_list.append(image_path)
                msk_list.append(cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), (CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE'])))
    
            if os.path.exists("temp_dir"):
                shutil.rmtree("temp_dir")
            os.makedirs("temp_dir", exist_ok=True)
            for img in img_path_list:
                temp_img = cv2.resize(cv2.imread(img), (CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE']))
                temp_img_list_path.append(os.path.join("temp_dir",img.split("/")[-1]))
                cv2.imwrite(os.path.join("temp_dir",img.split("/")[-1]), temp_img)
                
            cp_model_path = cellposesam_results[0][0]
            multiple_masks, _, _ = cellpose_inference(temp_img_list_path, cp_model_path)
    
    
            if os.path.exists("cellpose_mask_predictions"):
                shutil.rmtree("cellpose_mask_predictions")
            os.makedirs("/kaggle/working/cellpose_mask_predictions", exist_ok=True)
            df_test = pd.read_csv("results/test_data.csv")
            for idx, (mask, img_filename) in enumerate(zip(multiple_masks, df_test['image_filename'])):
                # Ensure mask is uint8 (0–255)
                mask_img = (mask.astype("uint8")) * 255
                
                # Save as PNG
                save_path = os.path.join("/kaggle/working/cellpose_mask_predictions", f"{img_filename}")
                cv2.imwrite(save_path, mask_img)
    
            zip_path = "/kaggle/working/cellpose_mask_predictions.zip"
            shutil.make_archive(zip_path.replace(".zip", ""), 'zip', "/kaggle/working/cellpose_mask_predictions")
            print(f"Saved masks")
    
    
            
            preds = np.array(multiple_masks)
            targets = np.array(msk_list)
            
            preds_bin   = (np.array(multiple_masks) > 0).astype(np.float32)
            targets_bin = (np.array(msk_list) > 127).astype(np.float32)
    
            
            cp_best_iou = compute_iou(preds_bin, targets_bin, threshold=0.5, smooth=1e-6)
            cp_best_dice = compute_dice(preds_bin, targets_bin, threshold=0.5, smooth=1e-6)
            cp_best_loss = np.nan
    
            
            test_results.append({
                "Model":     name,
                "Test Loss": round(cp_best_loss, 4),
                "Test IoU":  round(cp_best_iou,  4),
                "Test Dice": round(cp_best_dice, 4),
            })
            print(f"  {name:<40} Loss: {cp_best_loss:.4f} | "
                  f"IoU: {cp_best_iou:.4f} | Dice: {cp_best_dice:.4f}")
        except Exception as exc:
            print(f"  ❌ Test eval failed for {name}: {exc}")
            # continue
            
        if entry["Best Val IoU"] is None:
            print(f"  Skipping {name} (training failed).")
            continue

    ckpt_path = os.path.join(CONFIG["SAVE_DIR"],
                             f"{name.replace(' ', '_')}_best.pt")
    if not os.path.exists(ckpt_path):
        print(f"  ⚠️  Checkpoint not found for {name}, skipping test eval.")
        continue

    factory = name_to_factory.get(name)
    if factory is None:
        continue

    try:
        model = factory().to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        test_loss, test_iou, test_dice = evaluate(model, test_loader, device, criterion, name=name)
        test_results.append({
            "Model":     name,
            "Test Loss": round(test_loss, 4),
            "Test IoU":  round(test_iou,  4),
            "Test Dice": round(test_dice, 4),
        })
        print(f"  {name:<40} Loss: {test_loss:.4f} | "
              f"IoU: {test_iou:.4f} | Dice: {test_dice:.4f}")
    except Exception as exc:
        print(f"  ❌ Test eval failed for {name}: {exc}")
    finally:
        try:
            del model
        except NameError:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

test_df = pd.DataFrame(test_results)
print("\n✅ Test evaluation complete.")


════════════════════════════════════════════════════════════
  TEST SET EVALUATION
════════════════════════════════════════════════════════════
  Inception-UNet                           Loss: 0.2387 | IoU: 0.9524 | Dice: 0.9756
  UNet (Baseline)                          Loss: 0.0606 | IoU: 0.9658 | Dice: 0.9826
  ⚠️  micro-SAM unavailable (No module named 'micro_sam'). Using UNet surrogate.
  microSAM                                 Loss: 0.0805 | IoU: 0.9639 | Dice: 0.9816
2026-06-02 15:49:11,624 [INFO] ** TORCH CUDA version installed and working. **
2026-06-02 15:49:11,625 [INFO] >>>> using GPU (CUDA)
2026-06-02 15:49:13,839 [INFO] >>>> loading model /kaggle/working/models/cellpose_model
2026-06-02 15:49:14,701 [WARNING] channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
2026-06-02 15:50:17,409 [INFO] 44%|####3     | 38/87 [01:02<01:20,  1.65s/it]


In [ ]:
from IPython.display import display

# ── Validation summary ────────────────────────────────────────────────────────
val_df = (pd.DataFrame(results)
          .sort_values("Best Val IoU", ascending=False)
          .reset_index(drop=True))
val_df.index += 1
val_df.index.name = "Rank"

# ── Combined CSV: validation + test metrics ───────────────────────────────────
if not test_df.empty:
    combined_df = val_df.reset_index().merge(
        test_df, on="Model", how="left")
    combined_df = combined_df.sort_values("Best Val IoU", ascending=False)
else:
    combined_df = val_df.reset_index()

combined_csv = os.path.join(CONFIG["RESULTS_DIR"], "combined_results.csv")
combined_df.to_csv(combined_csv, index=False)
print(f"✅ Combined results CSV saved → {combined_csv}\n")

# ── Display validation table ──────────────────────────────────────────────────
print("─"*60)
print("  VALIDATION RESULTS")
print("─"*60)
display(val_df.style
    .background_gradient(
        subset=["Best Val Loss", "Best Val IoU", "Best Val Dice"],
        cmap="YlGn")
    .format({
        "Best Val Loss": "{:.4f}",
        "Best Val IoU":  "{:.4f}",
        "Best Val Dice": "{:.4f}",
    })
    .set_caption("Segmentation Model Benchmark — Validation Metrics"))

# ── Display test table ────────────────────────────────────────────────────────
if not test_df.empty:
    print("\n" + "─"*60)
    print("  TEST RESULTS")
    print("─"*60)
    display(test_df.style
        .background_gradient(
            subset=["Test Loss", "Test IoU", "Test Dice"],
            cmap="YlGn")
        .format({
            "Test Loss": "{:.4f}",
            "Test IoU":  "{:.4f}",
            "Test Dice": "{:.4f}",
        })
        .set_caption("Segmentation Model Benchmark — Test Metrics"))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

n_models = len(histories)
cols     = 3
rows     = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
axes      = axes.flatten()
cmap      = plt.cm.get_cmap("tab10")

for i, (name, hist) in enumerate(histories.items()):
    ax     = axes[i]
    color  = cmap(i % 10)
    epochs = [h["epoch"]    for h in hist]
    ious   = [h["val_iou"]  for h in hist]
    dices  = [h["val_dice"] for h in hist]
    losses = [h["val_loss"] for h in hist]

    ax2 = ax.twinx()
    ax.plot(epochs, ious,   label="Val IoU",  color=color,       lw=2)
    ax.plot(epochs, dices,  label="Val Dice", color=color, lw=2, ls="--")
    ax2.plot(epochs, losses, label="Val Loss", color="grey",      lw=1, ls=":")
    ax2.set_ylabel("Loss", fontsize=8, color="grey")
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.grid(True, alpha=0.3)

for ax in axes[n_models:]:
    ax.set_visible(False)

fig.suptitle("Validation Metrics per Epoch", fontsize=14,
             fontweight="bold", y=1.01)
plt.tight_layout()
curve_path = os.path.join(CONFIG["RESULTS_DIR"], "training_curves.png")
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {curve_path}")

In [ ]:
import seaborn as sns

df_plot = val_df.reset_index()[["Model", "Best Val IoU", "Best Val Dice"]]
df_melt = df_plot.melt(id_vars="Model", var_name="Metric", value_name="Score")

fig, ax = plt.subplots(figsize=(max(10, n_models * 1.5), 5))
sns.barplot(data=df_melt, x="Model", y="Score", hue="Metric",
            palette=["#2196F3", "#4CAF50"], ax=ax)

ax.set_ylim(0, 1.05)
ax.set_title("IoU & Dice Comparison Across Models",
             fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Metric", loc="lower right")
ax.grid(axis="y", alpha=0.3)

for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{h:.3f}",
                    (p.get_x() + p.get_width() / 2.0, h),
                    ha="center", va="bottom", fontsize=7.5)

plt.tight_layout()
bar_path = os.path.join(CONFIG["RESULTS_DIR"], "model_comparison.png")
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {bar_path}")

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
STD  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]


def unnormalize(t):
    return torch.clamp(t * STD + MEAN, 0, 1)


def visualise_predictions(name, factory, config, val_loader, n_samples=4):
    device    = torch.device(config["DEVICE"] if torch.cuda.is_available() else "cpu")
    model     = factory().to(device)
    os.makedirs(config["SAVE_DIR"], exist_ok=True)
    ckpt_path = os.path.join(config["SAVE_DIR"],
                             f"{name.replace(' ', '_')}_best.pt")
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    imgs_b, masks_b = next(iter(val_loader))
    imgs_b = imgs_b[:n_samples].to(device)
    masks_b = masks_b[:n_samples]

    with torch.no_grad():
        out   = model(imgs_b)
        if out.shape != masks_b.shape:
            out = F.interpolate(out, size=masks_b.shape[2:])
        preds = (torch.sigmoid(out).cpu() > 0.5).float()

    fig, axes = plt.subplots(n_samples, 3, figsize=(9, 3 * n_samples))
    if n_samples == 1:
        axes = axes[None]

    for i in range(n_samples):
        img_np  = unnormalize(imgs_b[i].cpu()).permute(1, 2, 0).numpy()
        mask_np = masks_b[i, 0].numpy()
        pred_np = preds[i, 0].numpy()

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title("Image"      if i == 0 else "")
        axes[i, 1].imshow(mask_np, cmap="gray")
        axes[i, 1].set_title("GT Mask"    if i == 0 else "")
        axes[i, 2].imshow(pred_np, cmap="gray")
        axes[i, 2].set_title("Prediction" if i == 0 else "")
        for ax in axes[i]:
            ax.axis("off")

    fig.suptitle(f"{name} — Sample Predictions", fontsize=13, fontweight="bold")
    plt.tight_layout()
    out_path = os.path.join(config["RESULTS_DIR"],
                            f"preds_{name.replace(' ', '_')}.png")
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Visualise the top-2 models by validation IoU
name_to_factory = {r[0]: r[2] for r in model_registry}
top2 = val_df.reset_index()["Model"].tolist()[:2]

for mname in top2:
    if mname in name_to_factory:
        visualise_predictions(mname, name_to_factory[mname],
                              CONFIG, val_loader)

In [ ]:
print("\n" + "═"*65)
print("  FINAL SEGMENTATION BENCHMARK RESULTS")
print("═"*65)

meta = {
    "Inception-UNet":                ("Andrusca et al.", 2026, "[16]"),
    "StarDist":                       ("Gros et al.",     2025, "[14]"),
    "DeGPR / FPN-ResNet18":          ("Awais et al.",    2025, "[17]"),
    "DECSEFE-Org / UNet++ DenseNet": ("Cicceri et al.",  2025, "[19]"),
    "microSAM":                       ("Archit et al.",   2025, "[12]"),
    "CellPoseSAM":                    ("Pachitariu et al.", 2025, "[20]"),
    "UNet (Baseline)":                ("Present Thesis",  2026, ""),
}

rows_out = []
for _, row in val_df.iterrows():
    m = row["Model"]
    auth, yr, ref = meta.get(m, ("—", "—", ""))
    rows_out.append({
        "Rank":          row.name,
        "Model":         m,
        "Author":        f"{auth} {yr}{ref}",
        "Val IoU ↑":     f"{row['Best Val IoU']:.4f}"  if row["Best Val IoU"]  else "—",
        "Val Dice ↑":    f"{row['Best Val Dice']:.4f}" if row["Best Val Dice"] else "—",
        "Val Loss ↓":    f"{row['Best Val Loss']:.4f}" if row["Best Val Loss"] else "—",
    })

final_df = pd.DataFrame(rows_out).set_index("Rank")

# Merge test metrics if available
if not test_df.empty:
    final_df = final_df.merge(test_df.rename(columns={
        "Test IoU":  "Test IoU ↑",
        "Test Dice": "Test Dice ↑",
        "Test Loss": "Test Loss ↓",
    }), on="Model", how="left")

display(final_df)

final_csv = os.path.join(CONFIG["RESULTS_DIR"], "final_benchmark_table.csv")
final_df.to_csv(final_csv)
print(f"\n✅ Saved → {final_csv}")
print(f"✅ All CSVs written to  {CONFIG['RESULTS_DIR']}/")